# 03: Converting API Data to Pandas DataFrame

## What You Will Learn
- Why use Pandas for API data
- How to convert a list of dicts to DataFrame
- Basic DataFrame inspection
- Simple data cleaning
- Export to CSV (for future use)

---

## Why Pandas for API Data?

You currently have a **list of dictionaries** like this:
```python
[
  {'city': 'Delhi', 'temp': 32, 'humidity': 40},
  {'city': 'London', 'temp': 15, 'humidity': 75}
]
```

**Pandas converts this to a table (DataFrame):**

| city | temp | humidity |
|------|------|----------|
| Delhi | 32 | 40 |
| London | 15 | 75 |

**Benefits:**
- Visualize data easily
- Sort, filter, analyze
- Export to CSV/Excel/Database
- Handle missing data
- Statistics with one line

## Step 1: Setup - Reuse Your API Function

In [1]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd  # New import for this notebook

# Load environment variables
load_dotenv()
api_key = os.getenv("OPENWEATHER_API_KEY")

# Reuse the safe function from previous notebook
def get_weather(city_name):
    """Fetch weather data for a city with error handling."""
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={api_key}&units=metric"
    
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return response.json()
        else:
            return None
    except:
        return None

print("Setup complete. Pandas version:", pd.__version__)

Setup complete. Pandas version: 3.0.2


## Step 2: Fetch Data for Multiple Cities

In [2]:
# Fetch weather for multiple cities
cities = ["Delhi", "Mumbai", "London", "Tokyo", "New York", "Paris", "Sydney"]

weather_list = []

for city in cities:
    print(f"Fetching {city}...", end=" ")
    data = get_weather(city)
    
    if data:
        weather_list.append({
            'city': data['name'],
            'country': data['sys']['country'],
            'temperature': data['main']['temp'],
            'feels_like': data['main']['feels_like'],
            'humidity': data['main']['humidity'],
            'pressure': data['main']['pressure'],
            'description': data['weather'][0]['description'],
            'wind_speed': data['wind']['speed']
        })
        print("✓")
    else:
        print("✗")

print(f"\nCollected {len(weather_list)} records")

Fetching Delhi... ✓
Fetching Mumbai... ✓
Fetching London... ✓
Fetching Tokyo... ✓
Fetching New York... ✓
Fetching Paris... ✓
Fetching Sydney... ✓

Collected 7 records


## Step 3: Convert List to DataFrame

**The magic one-liner:**
```python
df = pd.DataFrame(list_of_dictionaries)
```

Pandas automatically:
- Creates columns from dictionary keys
- Creates rows from each dictionary
- Handles data types intelligently

In [3]:
# Convert list of dicts to DataFrame
df = pd.DataFrame(weather_list)

# Display the DataFrame
print("DataFrame created!\n")
df

DataFrame created!



,city,country,temperature,feels_like,humidity,pressure,description,wind_speed
0,Delhi,IN,31.05,30.98,40,1005,haze,0.00
1,Mumbai,IN,29.99,35.01,70,1008,haze,3.09
2,London,GB,16.37,15.21,44,1021,scattered clouds,4.12
3,Tokyo,JP,15.91,15.80,86,1015,clear sky,1.03
4,New York,US,17.05,16.64,70,1018,broken clouds,4.63
5,Paris,FR,18.70,18.30,64,1021,broken clouds,6.17
6,Sydney,AU,11.24,10.46,78,1020,clear sky,3.60


## Step 4: Inspect Your DataFrame

Key methods to understand your data:

In [4]:
# Basic info about the DataFrame
print("=== DataFrame Info ===")
df.info()

print("\n=== First 3 Rows ===")
print(df.head(3))

print("\n=== Last 2 Rows ===")
print(df.tail(2))

print("\n=== Statistical Summary ===")
print(df.describe())

=== DataFrame Info ===
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   city         7 non-null      str    
 1   country      7 non-null      str    
 2   temperature  7 non-null      float64
 3   feels_like   7 non-null      float64
 4   humidity     7 non-null      int64  
 5   pressure     7 non-null      int64  
 6   description  7 non-null      str    
 7   wind_speed   7 non-null      float64
dtypes: float64(3), int64(2), str(3)
memory usage: 580.0 bytes

=== First 3 Rows ===
     city country  temperature  feels_like  humidity  pressure  \
0   Delhi      IN        31.05       30.98        40      1005   
1  Mumbai      IN        29.99       35.01        70      1008   
2  London      GB        16.37       15.21        44      1021   

        description  wind_speed  
0              haze        0.00  
1              haze        3.09  
2  scattered clo

## Step 5: Selecting Columns

Access specific columns for analysis:

In [5]:
# Select a single column (returns Series)
print("Temperatures:")
print(df['temperature'])

print("\n\nSelect multiple columns:")
# Select multiple columns (returns DataFrame)
print(df[['city', 'country', 'temperature', 'description']])

Temperatures:
0    31.05
1    29.99
2    16.37
3    15.91
4    17.05
5    18.70
6    11.24
Name: temperature, dtype: float64


Select multiple columns:
       city country  temperature       description
0     Delhi      IN        31.05              haze
1    Mumbai      IN        29.99              haze
2    London      GB        16.37  scattered clouds
3     Tokyo      JP        15.91         clear sky
4  New York      US        17.05     broken clouds
5     Paris      FR        18.70     broken clouds
6    Sydney      AU        11.24         clear sky


## Step 6: Filtering Data

Find cities that meet certain conditions:

In [6]:
# Cities hotter than 25°C
hot_cities = df[df['temperature'] > 25]
print("Cities hotter than 25°C:")
print(hot_cities[['city', 'temperature']])

print("\n\nCities with high humidity (>70%):")
humid_cities = df[df['humidity'] > 70]
print(humid_cities[['city', 'humidity', 'description']])

Cities hotter than 25°C:
     city  temperature
0   Delhi        31.05
1  Mumbai        29.99


Cities with high humidity (>70%):
     city  humidity description
3   Tokyo        86   clear sky
6  Sydney        78   clear sky


## Step 7: Sorting Data

In [7]:
# Sort by temperature (hottest first)
print("Cities sorted by temperature (hottest first):")
df_sorted = df.sort_values('temperature', ascending=False)
print(df_sorted[['city', 'country', 'temperature', 'description']])

Cities sorted by temperature (hottest first):
       city country  temperature       description
0     Delhi      IN        31.05              haze
1    Mumbai      IN        29.99              haze
5     Paris      FR        18.70     broken clouds
4  New York      US        17.05     broken clouds
2    London      GB        16.37  scattered clouds
3     Tokyo      JP        15.91         clear sky
6    Sydney      AU        11.24         clear sky


## Step 8: Export to CSV

Save your data for later analysis or sharing:

In [8]:
# Save to CSV file
output_path = "../weather_data.csv"
df.to_csv(output_path, index=False)

print(f"Data saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path)} bytes")

Data saved to: ../weather_data.csv
File size: 399 bytes


---

## Summary

✅ **You learned:**
- How to convert API data to DataFrame
- `pd.DataFrame(list_of_dicts)` is the key
- How to inspect data with `.head()`, `.info()`, `.describe()`
- How to select and filter columns
- How to sort data
- How to export to CSV

**Next Steps for Your Project:**
1. Create main project notebook
2. Fetch weather + air quality together
3. Store in SQL database
4. Build analysis and dashboards

**You're ready to start the main project!**